# LeapQuant KDA 教程 (3/5): KDA 状态的 int8 快照 —— 同一套量化器照样用

LeapQuant 的核心动作: 每 $W$ 个 token 把状态 $S$ $[V,K]$ 过一遍 **rank-r 补偿 + 双尺度 int8**
快照 (论文 eq.5–9), 把 fp32 状态的存放开销压到 ~1.19 B/elem。

**关键问题**: 这套量化器是为 GDN 状态调的 —— KDA 状态 (逐列衰减 → 列范数分层) 还能量化好吗?

答案: 能。量化器只看矩阵的**低秩 + 离群点**结构, 与门怎么来的无关。本册在真实规模
(Kimi KDA 头几何, $128 \times 128$) 上验证。

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发仓 (kernel / 量化器所在)
torch.npu.set_device(0)
DEV = torch.device("npu:0")
torch.manual_seed(7)
print("device:", torch.npu.get_device_name(0))


from vllm_ascend.quantization.methods.mamba_state.leapquant.state import (
    start_basis, fit_compensators, quantize_residual, reconstruct_state, leap_window_flush,
)

H, V, K, R = 8, 128, 128, 4
WINDOW = 16

/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


device: Ascend910_9382


/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:13: FutureWarning: triton.language.extra.cann.extension.atan2 is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.atan2 instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.atan2")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:35: FutureWarning: triton.language.extra.cann.extension.isfinited is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.isfinited instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.isfinited")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:49: FutureWarning: triton.language.extra.cann.extension.finitef is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.finitef instead.
  @_deprec

INFO 11-11 16:22:24 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 11-11 16:22:24 [__init__.py:54] - ascend -> vllm_ascend:register


INFO 11-11 16:22:24 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 11-11 16:22:24 [__init__.py:272] Platform plugin ascend is activated


INFO 11-11 16:22:24 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


/usr/local/python3.11.10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 滚一个真实的 KDA 边界状态

用教程 1 的递推 (fp32) 滚 `WINDOW` 个 token, 拿到每个头的边界状态 $S_h \in \mathbb{R}^{128 \times 128}$
—— 这就是窗口边界时要"拍照"的对象。对照同样滚一个 GDN 状态。

In [2]:
def roll_state(per_k: bool, T=WINDOW, seed=11):
    torch.manual_seed(seed)
    g_vec = -0.1 - 0.6 * torch.rand(H, K, device=DEV)          # KDA: 每列一个门
    g_sca = g_vec.mean(dim=1)                                  # GDN: 同均值的标量门
    S = torch.zeros(H, V, K, device=DEV)
    for _ in range(T):
        k = torch.nn.functional.normalize(torch.randn(H, K, device=DEV), dim=-1)
        v = torch.randn(H, V, device=DEV)
        beta = (torch.rand(H, 1, device=DEV) * 0.5 + 0.5)
        g = g_vec if per_k else g_sca[:, None]
        S = S * torch.exp(g)[:, None, :]                       # per_k=False 时 g 沿 K 广播
        sk = torch.einsum("hvk,hk->hv", S, k)
        S = S + torch.einsum("hv,hk->hvk", (v - sk) * beta, k)
    return S

S_kda = roll_state(per_k=True)
S_gdn = roll_state(per_k=False)
print("KDA 状态范数:", f"{S_kda.norm():.2f}", " GDN 状态范数:", f"{S_gdn.norm():.2f}")
print("KDA 列范数 (头 0, 前 8 列):", [f"{x:.2f}" for x in S_kda[0].norm(dim=0)[:8].tolist()])

[W1111 16:22:31.134962073 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


KDA 状态范数: 35.20  GDN 状态范数: 32.25
KDA 列范数 (头 0, 前 8 列): ['0.72', '1.10', '1.11', '1.40', '0.87', '2.18', '1.53', '0.76']


## 2. 能量集中吗 —— 奇异值谱

LeapQuant 能压的前提: 状态能量集中在少数奇异方向 (窗口只有 16 个 token, 秩 ≤ 16,
而 top-$r$ ($r{=}4$) 通常已拿走大部分)。KDA 的列范数分层会不会破坏这一点?

In [3]:
def top_r_energy(S, r=R):
    s = torch.linalg.svdvals(S.cpu())                                # 奇异值, 降序
    return (s[:r] ** 2).sum() / (s ** 2).sum()

frac_kda = [top_r_energy(S_kda[h]).item() for h in range(H)]
frac_gdn = [top_r_energy(S_gdn[h]).item() for h in range(H)]
print(f"top-{R} 能量占比   KDA: 均值 {sum(frac_kda)/H:.3f}  范围 [{min(frac_kda):.3f}, {max(frac_kda):.3f}]")
print(f"              GDN: 均值 {sum(frac_gdn)/H:.3f}  范围 [{min(frac_gdn):.3f}, {max(frac_gdn):.3f}]")

top-4 能量占比   KDA: 均值 0.882  范围 [0.845, 0.907]
              GDN: 均值 0.960  范围 [0.942, 0.972]


## 3. 同一套流水线: 基 → 补偿对 → 残差 int8 → 重建

与 GDN 教程 3 完全相同的四步 (论文 eq.5–9), 输入换成 KDA 状态, 一行不改:

In [4]:
blocks = S_kda.clone()
q0 = start_basis(V, R, device=DEV, dtype=torch.float32)        # 固定起始基 (逐位可复现)
u, q = fit_compensators(blocks, q0, R)                         # 一轮子空间迭代追 top-r
codes, s_k, s_v = quantize_residual(blocks, u, q)              # 残差双尺度 int8 (eq.7)
S_hat = reconstruct_state(codes, s_k, s_v, u, q)               # 重建 (eq.8-9)

print(f"codes int8 ∈ [{codes.min()}, {codes.max()}],  scales: s_k均值 {s_k.mean():.3f}, s_v均值 {s_v.mean():.3f}")
err = ((S_hat - blocks).norm() / blocks.norm()).item()
print(f"LeapQuant 重建 rel err = {err:.4f}")

codes int8 ∈ [-127, 127],  scales: s_k均值 0.156, s_v均值 0.587
LeapQuant 重建 rel err = 0.0024


/workspace/y00924007/vllm-ascend/vllm_ascend/quantization/methods/mamba_state/leapquant/state.py:75: UserWarning: Cannot create tensor with interal format while allow_internel_format=False, tensor will be created with base format. (Triggered internally at ../torch_npu/csrc/aten/common/TensorFactories.cpp:437.)
  return q0.to(device=device, dtype=dtype).contiguous()


## 4. 对照: naive int8 (整块一个尺度)

In [5]:
err_naive = 0.0
for h in range(H):
    Sh = blocks[h]
    scale = Sh.abs().max() / 127
    q8 = (Sh / scale).round().clamp(-127, 127).to(torch.int8)
    rec = q8.float() * scale
    err_naive += ((rec - Sh) / Sh.norm()).norm() ** 2
err_naive = err_naive.sqrt().item()
print(f"naive int8  rel err = {err_naive:.4f}   vs  LeapQuant = {err:.4f}   "
      f"({err_naive / err:.2f}× 更好)")

naive int8  rel err = 0.0479   vs  LeapQuant = 0.0024   (20.32× 更好)


## 小结 (KDA 教程 3)

- KDA 边界状态与 GDN 一样低秩 (top-4 能量占比同量级) —— 列范数分层不破坏集中性;
- `start_basis`/`fit_compensators`/`quantize_residual`/`reconstruct_state` **一行不改**直接吃 KDA 状态;
- `leap_window_flush` 打包版同样适用 (教程 4 直接用它做窗口扫描)。